**Vision-Based Gaze Estimation | Presented by: Reza Saadatyar (2026)**

**04 | CNN and Transfer-Learning Gaze Estimation**

![Study-specific MPIIGaze research workflow and evidence boundaries](../../images/04_cnn_and_transfer_learning.svg)

**Objective:** Train a small grayscale CNN and a lightweight pretrained ResNet-18 on the same source-matched eye patches and participant folds as the classical baselines.

**Inputs:** Verified Notebook 02 sample index, outer/inner Fold assignments and an original-source image cache kept outside Git.

**Evaluation boundary:** Development uses inner-Validation participants for checkpoint selection. **Outer Test scoring is disabled by default**; enabling it after model development is a distinct scientific step. Numerical plots require actual model execution.

**00. Colab Repository Setup**

In [ ]:
# ======================== 00. Colab Repository Setup =========================

# -------------------------- 1. Repository Checkout ---------------------------
from pathlib import Path
import subprocess
repo_url="https://github.com/RezaSaadatyar/Eye-Tracking-Neurocognitive-Intelligence.git"
repo_path=Path("/content")/repo_url.rsplit("/",1)[-1].removesuffix(".git")
if not repo_path.exists():
    subprocess.run(["git","clone","-q",repo_url,str(repo_path)],check=True)

# ---------------------------- 2. Open Repository -----------------------------
%cd {repo_path}

**01. Imports and Research Environment**

In [ ]:
# =================== 01. Imports and Research Environment ====================

# ------------------------ 1. Study Environment Setup -------------------------
from python_modules.core.colab_setup import setup_colab
root=setup_colab("vision_based_gaze_estimation")

# ---------------------------- 2. Standard Library ----------------------------
import random
import sys

# ----------------- 3. Scientific Computing and Data Analysis -----------------
import numpy as np
import pandas as pd

# ------------------- 4. Machine Learning and Deep Learning -------------------
import torch
from gaze_models import train_gaze_network, predict_network
from gaze_geometry import angular_error_degrees
from mpiigaze_data import restore_indexed_eye_cache
from gaze_evaluation import (load_image_cache, outer_fold_indices,
                             person_angular_metrics)

# ------------------- 5. Visualisation and Notebook Display -------------------
import matplotlib.pyplot as plt
from IPython.display import display
from core.plot_style import plot_style, save_figure

# ------------------- 6. Shared and Study Research Utilities ------------------
from core.computer_device import computer_device
from core.output_download import download_outputs_from_colab

# ------------------- 7. Reproducibility and Compute Device -------------------
SEED=17
random.seed(SEED)
np.random.seed(SEED)
device=computer_device(min_vram_gb=None)

# --------------------- 8. Output Directory and Setup -------------------------
out_dir=root/"vision_based_gaze_estimation"/"output"
cache_dir=root/"data"/"processed"/"mpiigaze"
cache_path=cache_dir/"02_normalised_eye_cache.npz"
checkpoint_dir=cache_dir/"checkpoints"
checkpoint_dir.mkdir(parents=True,exist_ok=True)
plot_style(font_family="DejaVu Sans",title_size=12.5,
 title_weight="normal",label_size=11.5,label_weight="normal",
 tick_size=11,legend_size=10,figure_dpi=100,figure_size=(9,4))
print("Study outputs:",out_dir)
print("Local-only model checkpoints:",checkpoint_dir)
print("Python:",sys.version.split()[0])

**02. Check Original Images and Fold Identities**

Neural and classical models consume the *same* Notebook 02 image–target table and participant-held-out folds. If the image cache is absent, restore the exact saved Stage-02 source rows from the official archive in Colab. This does not resample images, alter fold assignments or fit any model.

In [ ]:
# ================= 02. Verify Current Image and Fold Inputs ==================

required=["02_sample_index.csv","02_fold_assignments.csv"]
missing=[x for x in required if not (out_dir/x).is_file()]
if missing:
    raise FileNotFoundError(f"Notebook 02 outputs missing: {missing}")
# Recover the exact Stage-02 sample rows if this runtime is new.
if not cache_path.is_file():
    restore_indexed_eye_cache(
        root / "data" / "raw" / "mpiigaze",
        out_dir / "02_sample_index.csv", cache_path)
index,images,targets=load_image_cache(
 out_dir/"02_sample_index.csv",cache_path)
folds=pd.read_csv(out_dir/"02_fold_assignments.csv",
 dtype={"participant_id":str,"fold_id":str,"outer_test_person":str})
if folds.duplicated(["fold_id","participant_id"]).any() or folds.empty:
    raise ValueError("Outer-fold source identities are repeated or missing")
print("Matching original image examples:",len(index))
print("Independent people:",index.participant_id.nunique())
print("Available outer folds:",folds.fold_id.nunique())

**03. Select Development Mode and Model Configurations**

A short **development run** verifies the model fitting, checkpointing and inner validation on two outer-person configurations without exposing their Test labels. Set `RUN_OUTER_TEST = True` only for a separately authorised final evaluation after architecture and epoch rules are fixed. A complete LOPO evaluation would require every outer fold, not two.

In [ ]:
# =================== 03. Configure Nested Neural Training ====================

# --------------------- 1. Explicit Compute and Test Settings -----------------
RUN_OUTER_TEST=False
MAX_OUTER_FOLDS=2 if not RUN_OUTER_TEST else None
MODEL_NAMES=("small_cnn","resnet18")
PRETRAINED_RESNET=True
EPOCHS=8
PATIENCE=3
BATCH_SIZE=128
LEARNING_RATE=1e-3

# ------------------------ 2. Preserve Test Boundaries ------------------------
all_folds=sorted(folds.fold_id.unique())
active_folds=(all_folds if MAX_OUTER_FOLDS is None
              else all_folds[:MAX_OUTER_FOLDS])
if RUN_OUTER_TEST and len(active_folds)!=len(all_folds):
    raise ValueError("A final outer LOPO run requires all outer participants")
if not RUN_OUTER_TEST:
    prior=[f for f in ("04_neural_outer_predictions.csv",
                       "04_neural_outer_metrics.csv") if (out_dir/f).exists()]
    if prior:
        raise RuntimeError("Older Test files exist; preserve the current evidence "
                           "and use a clean development checkout: "+str(prior))
print("Current mode:","outer Test" if RUN_OUTER_TEST else "inner Validation development")
print("Outer fold configurations:",len(active_folds),"/",len(all_folds))
print("Model families:",MODEL_NAMES)

**04. Train Small CNN and ResNet-18 Within Development Roles**

Each model is trained only on the fold's inner-training people. Early stopping selects its checkpoint based on angular error for different inner-Validation people. The pretrained ResNet-18 branch may download standard ImageNet weights, not MPIIGaze observations; that provenance is recorded separately.

In [ ]:
# =============== 04. Train Fold-Specific CNN and ResNet Models ===============

# ---------------------- 1. Fit Inner Development Models ----------------------
history_rows,inner_metrics,outer_prediction_rows,checkpoint_rows=[],[],[],[]
for fold_i,fold_id in enumerate(active_folds):
    roles=outer_fold_indices(index,folds,fold_id)
    i_train,i_val,i_test=(roles[k] for k in ("train","validation","test"))
    print("Fold:",fold_id,"training people:",index.iloc[i_train].participant_id.nunique(),
          "inner Validation people:",index.iloc[i_val].participant_id.nunique())
    for model_name in MODEL_NAMES:
        pretrained=bool(model_name=="resnet18" and PRETRAINED_RESNET)
        network,history,choice=train_gaze_network(
            model_name,images,targets,i_train,i_val,device=device,
            epochs=EPOCHS,batch_size=BATCH_SIZE,
            seed=SEED+fold_i,pretrained=pretrained,
            learning_rate=(1e-4 if pretrained else LEARNING_RATE),
            patience=PATIENCE)
        for step in history:
            history_rows.append({"fold_id":fold_id,"model":model_name,**step})
        inner_metrics.append({"fold_id":fold_id,"model":model_name,
                              "n_fit_people":index.iloc[i_train].participant_id.nunique(),
                              "n_val_people":index.iloc[i_val].participant_id.nunique(),**choice})
        path=checkpoint_dir/f"04_{model_name}_{fold_id}.pt"
        torch.save({"state_dict":network.state_dict(),"fold_id":fold_id,
                    "model":model_name,"settings":choice},path)
        checkpoint_rows.append({"fold_id":fold_id,"model":model_name,
            "local_relative_path":path.relative_to(root).as_posix(),
            **choice})
        if RUN_OUTER_TEST:
            predicted=predict_network(network,images,i_test,device)
            part=index.iloc[i_test][["sample_id","participant_id"]].copy()
            part["fold_id"]=fold_id
            part["model"]=model_name
            part[["target_x","target_y","target_z"]]=targets[i_test]
            part[["pred_x","pred_y","pred_z"]]=predicted
            outer_prediction_rows.append(part)
        print(model_name,"| best epoch:",choice["best_epoch"],
              "| inner Validation angular error:",
              round(choice["inner_validation_mean_angular_error_deg"],3))

history_table=pd.DataFrame(history_rows)
inner_table=pd.DataFrame(inner_metrics)
checkpoints=pd.DataFrame(checkpoint_rows)
display(inner_table)

**05. Examine Inner Validation and Resource Requirements**

This section compares **inner-Validation learning curves**, not Test performance. Pretrained ImageNet weights do not constitute gaze-trained checkpoints; only the fold-specific fitted checkpoint is used for any eventual Test prediction.

In [ ]:
# =============== 05. Inspect CNN Learning and Inner Validation ===============

fig,ax=plt.subplots(figsize=(10,4.6))
for name,group in history_table.groupby("model"):
    epoch_curve=group.groupby("epoch").inner_validation_mean_angular_error_deg.mean()
    ax.plot(epoch_curve.index,epoch_curve.values,marker="o",label=name)
ax.set(xlabel="Epoch",ylabel="Inner Validation mean angular error (degrees)",
 title="CNN development learning curves (not outer-Test scores)")
ax.legend()
ax.grid(alpha=0.2)
fig.tight_layout()
save_figure(fig,out_dir/"04_validation_curves.png")
plt.show()
print("Recorded model checkpoints stay under Git-ignored data/processed/")

**06. Save Model Evidence and Optional Final Predictions**

Development saves fold-specific training history and the inner selected checkpoint properties. Only the explicit final mode saves outer participant predictions. Stage 05 must distinguish partial development runs from full LOPO.

In [ ]:
# =============== 06. Save Neural Development and Test Evidence ===============

# ------------------------ 1. Save Development Tables -------------------------
history_table.to_csv(out_dir / "04_training_history.csv", index=False)
inner_table.to_csv(out_dir / "04_inner_validation_metrics.csv", index=False)
checkpoints.to_csv(out_dir / "04_model_checkpoints.csv", index=False)

# ------------------------ 2. Save Model Configuration ------------------------
configuration = pd.DataFrame([
    {
        "fold_id": fold_id, "model": model_name,
        "run_mode": ("final_outer_lopo" if RUN_OUTER_TEST
                     else "development_inner_validation"),
        "all_outer_folds": len(all_folds),
        "evaluated_folds": len(active_folds),
        "imagenet_pretrained": bool(model_name == "resnet18" and PRETRAINED_RESNET),
        "epochs_max": EPOCHS, "patience": PATIENCE, "batch_size": BATCH_SIZE,
        "learning_rate": (1e-4 if model_name == "resnet18" and PRETRAINED_RESNET
                          else LEARNING_RATE),
        "seed": SEED + fold_i, "test_used_for_model_selection": False,
    }
    for fold_i, fold_id in enumerate(active_folds)
    for model_name in MODEL_NAMES
])
configuration.to_csv(out_dir / "04_run_configuration.csv", index=False)

# ------------------------ 3. Save Outer Test Evidence ------------------------
if RUN_OUTER_TEST:
    predictions=pd.concat(outer_prediction_rows,ignore_index=True)
    predictions,person_metrics=person_angular_metrics(predictions)
    if predictions.groupby("model").sample_id.nunique().min()!=len(index):
        raise ValueError("Full outer LOPO predictions lack source sample coverage")
    predictions.to_csv(out_dir/"04_neural_outer_predictions.csv",index=False)
    person_metrics.to_csv(out_dir/"04_neural_outer_metrics.csv",index=False)
    print("Final outer-person predictions:",len(predictions))
else:
    print("Outer Test not scored; no Test prediction table written")

**07. Interpretation and Output Download**

The development run is for model readiness and inner Validation, not an independent generalisation estimate. Full LOPO inference requires a separately authorised final run without Test-informed tuning. The output ZIP contains the current stage's tables and plots, not its source images or weights.

In [ ]:
# ================== 07. Interpret Neural Results and Export ==================

needed=["04_training_history.csv","04_inner_validation_metrics.csv",
 "04_model_checkpoints.csv","04_run_configuration.csv",
 "04_validation_curves.png"]
if RUN_OUTER_TEST:
    needed+=["04_neural_outer_predictions.csv","04_neural_outer_metrics.csv"]
missing=[x for x in needed if not (out_dir/x).is_file()]
if missing:
    raise FileNotFoundError(f"Notebook 04 stage files missing: {missing}")
print("Recorded inner Validation model/fold configurations:",len(inner_table))
print("No new Test comparison has been used for model selection")
print("Next: 05_cross_person_evaluation_and_robustness.ipynb")
(out_dir / "04_run_configuration.json").unlink(missing_ok=True)
download_outputs_from_colab("vision_based_gaze_estimation/output","04")